# LangChain

### What is LangChain?
An open-source orchestration framework designed to simplify building applications powered by Large Language Models (LLMs).

---

## Core Architectural Concepts

| Concept | Definition | Role in LangChain |
| :--- | :--- | :--- |
| **Component** | Modular functional unit | Performs a specific job (e.g., `PromptTemplate`, `ChatModel`, `OutputParser`). |
| **Interface** | Standardized interaction protocol | Unified contract shared by components (e.g., Runnable protocol via `.invoke()`, `.stream()`). |
| **Abstraction** | Encapsulation of complexity | Hides provider-specific APIs (OpenAI, Ollama, Qdrant) behind clean, interchangeable interfaces. |
| **Chain** | Pipeline composition | Connects components sequentially to execute an end-to-end workflow (`Prompt → LLM → Parser`). |

---

### Architecture & Execution Flow

```text
              LangChain Framework
                       │
       ┌───────────────┼───────────────┐
       ↓               ↓               ↓
    Prompt            LLM            Parser
  (Component)     (Component)     (Component)
       │               │               │
       └───────────────┼───────────────┘
                       │
                   Interface
                       │ (Runnable Protocol)
                       ↓
                   .invoke()
                       │
                       ▼
                     Chain
                       │ (Pipeline Composition)
                       ▼
            Prompt  →  LLM  →  Parser
```

> **Engineering Note:**  
> Abstraction is embedded across LangChain's entire design. It decouples high-level application logic from low-level provider implementations—allowing you to swap underlying models, embeddings, or vector stores without rewriting pipeline logic.

## 1. Chat Models
A **ChatModel** is a core component that takes a sequence of messages as input and returns a structured chat response (`AIMessage`).

```bash
# Provider package installation
uv add langchain-anthropic
```

In [31]:
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv, find_dotenv
import os 

load_dotenv(find_dotenv())
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

# Initialize ChatModel Component
llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    temperature=0,
    max_tokens=100,
    api_key=anthropic_key
)

# Unified Interface invocation
response = llm.invoke("What is FastAPI?")
print(response)

content='# FastAPI\n\nFastAPI is a modern, fast web framework for building APIs with Python. Here are its key characteristics:\n\n## Core Features\n\n- **Fast Performance**: One of the fastest Python frameworks, comparable to Node.js and Go\n- **Easy to Learn**: Intuitive syntax based on standard Python type hints\n- **Automatic Documentation**: Auto-generates interactive API docs (Swagger UI and ReDoc)\n- **Built-in Validation**: Uses Pyd' additional_kwargs={} response_metadata={'id': 'msg_011CfZkvLnbr9FYCRBgZGGCK', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'max_tokens', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 12, 'output_tokens': 100, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'diagnostics': None, 'model

### Understanding the Output: `AIMessage` Object

When calling `.invoke()`, LangChain returns a standardized **`AIMessage`** object rather than a plain string.

```text
AIMessage Object
   │
   ├── .content            ──▶ The generated text (string)
   ├── .usage_metadata     ──▶ Token tracking (input, output, total)
   ├── .response_metadata  ──▶ Provider metadata (model, stop_reason)
   └── .id                 ──▶ Unique run identifier
```

| Attribute | Type | Description |
| :--- | :--- | :--- |
| **`content`** | `str` | The actual text answer generated by the model. |
| **`usage_metadata`** | `dict` | Token count breakdown: `input_tokens`, `output_tokens`, `total_tokens`. |
| **`response_metadata`**| `dict` | Specific provider info: `stop_reason` (`end_turn` vs `max_tokens`), model name. |
| **`id`** | `str` | Unique execution tracking ID (useful for LangSmith tracing). |

In [36]:
# Inspecting AIMessage attributes step-by-step:

# 1. Extract the text response
print("=== 1. Content ===")
print(response.content)

# 2. Inspect token usage (Crucial for cost & latency monitoring)
print("\n=== 2. Usage Metadata ===")
print("Input Tokens :", response.usage_metadata.get("input_tokens")) # return None if key not found
print("Output Tokens:", response.usage_metadata.get("output_tokens"))
print("Total Tokens :", response.usage_metadata.get("total_tokens"))
print("useage_metadata ojbect :", response.usage_metadata)

# 3. Check why the model stopped generating
print("\n=== 3. Stop Reason ===")
print(response.response_metadata.get("stop_reason"))

=== 1. Content ===
Your favorite programming language is Python, as you mentioned in your first message.

=== 2. Usage Metadata ===
Input Tokens : 43
Output Tokens: 18
Total Tokens : 61
useage_metadata ojbect : {'input_tokens': 43, 'output_tokens': 18, 'total_tokens': 61, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}

=== 3. Stop Reason ===
end_turn


## 2. Message Types in LangChain (`langchain_core.messages`)

Instead of passing raw strings, Chat Models operate on structured **Message objects** representing conversation roles:

| Message Type | Role | Purpose |
| :--- | :--- | :--- |
| **`SystemMessage`** | System / Developer | Sets behavior, tone, persona, constraints, and instructions for the model. |
| **`HumanMessage`** | User | Input coming from the human/user. |
| **`AIMessage`** | Assistant | The model's response. Also used to reconstruct conversation history. |

```text
Chat Conversation Flow:
[ SystemMessage (Persona) ➔ HumanMessage (User question) ➔ AIMessage (Model answer) ➔ HumanMessage (Follow-up) ]
```

In [34]:
from langchain_core.messages import HumanMessage

# 1. Invoking with a HumanMessage (passed inside a list of messages)
message = HumanMessage(content="What is FastAPI in 10 words?")
response = llm.invoke([message]) # must be in list

print("Message Type:", message.type)  # human
print("Response:", response.content)

Message Type: human
Response: # FastAPI

Modern Python web framework for building APIs quickly.


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

# 2. Steering model behavior using SystemMessage
messages = [
    SystemMessage(content="You are an expert Python tutor. Always reply in one short, enthusiastic sentence with an emoji."),
    HumanMessage(content="What is FastAPI?")
]

response = llm.invoke(messages)
print(response.content)

In [35]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# 3. Multi-turn conversation (Simulating Memory & Context History)
chat_history = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="My favorite programming language is Python."),
    AIMessage(content="That's awesome! Python is versatile and clean."),
    HumanMessage(content="What was my favorite language?")  # The model understands the context!
]
# ainvoke -> async await
response = await llm.ainvoke(chat_history)
print("Response with Context History:")
print(response.content)

Response with Context History:
Your favorite programming language is Python, as you mentioned in your first message.


## 3. Provider Abstraction: Swapping Models (Google Gemini)

One of LangChain's primary architectural advantages is **model provider abstraction**. Because all chat models adhere to the standardized `BaseChatModel` interface and Runnable protocol (`.invoke()`, `.ainvoke()`, `.stream()`), switching providers requires changing **only the model initialization**—downstream application logic, message formats, and prompt handling remain identical.

```bash
# Provider package installation
uv add langchain-google-genai
```

> **Key Engineering Considerations for Gemini Models:**
> 1. **Thinking / Reasoning Tokens**: Modern Gemini models (such as `gemini-3.5-flash`) utilize internal reasoning tokens before generating output. Setting an overly small `max_tokens` limit (e.g., 50–100) will exhaust the budget on reasoning alone, causing the visible response to be truncated. Provide an adequate token budget (e.g., `500+`).
> 2. **Response Content Representation**: In `langchain-google-genai`, `response.content` may be returned either as a plain string or as a list of content blocks (`[{"type": "text", "text": "..."}]`) reflecting Gemini's multimodal structure.

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())
gemini_key = os.getenv("GEMINI_API_KEY")

# 1. Initialize Gemini ChatModel Component
llm_gemini = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=gemini_key,
    temperature=0,
    max_tokens=500  # Token headroom for reasoning + generation
)

# 2. Unified Interface: identical .invoke() call as Claude
response_gemini = llm_gemini.invoke("What is FastAPI in one sentence?") 
print(response_gemini.content)

[{'type': 'text', 'text': '**FastAPI** is a modern, high-performance web framework for building APIs with Python, designed to be fast to code and easy to use based on standard Python type hints.', 'extras': {'signature': 'EssPCsgPAWkUfRP6ajDbOX/7qqLcCo4jA5ik2SzArDfN2ZIGWO1jBR2kAtXSvK5HsNo7gp2hjcmEYrN8p07MN9eugc0/fRTUTPSTzPlB0ZXCHCj795/ZXOdZ3nvMvfeJ1cvxCPi8PpFP5SQeExywcuUvh2VEvUL4WDgQZ0R7wideb3OACDTxResxQnipULGHznMSa4hRoRlTvApmo0f4SK9XgsPKgf6uuz/0u5DdePs5muT5ssOdJvi1sz7MZ4R+efn1KXKv6jC6jA9GdT5kXcefCKjQjUBmpeeazpPiocZ5Qx24ucK79oD1o4YBHtblK0BZpVwbtsSrDVwftLW7U6+CMdbPG66rwnw4ITe+m4KJkoUpfFKNJ70M3D6zu6lj/zxNGk7uOWZyG8MJmqIDLAYPLJhQzgXCbkaxFYmd4rtGguQtPNJsAbRpJlI5FEEB0M0l6wv1xCY4MwMjE2Dkfik4DSAKMb5Nah8VgbE3ygfq9rNBkGiTKKlA0U0ok3R4jt2rPKEywJLYLEozxNh0k3YbBrGyf+NGg34TKXQBYzFR/NVRUfKySpPH6/XJDqCD7kABAaPJC5GMsvixa5G/oBFa+3fghWWuWceOplI6UO19DyIKYyYGtBXX8ShENSJFkeGeuiO0KcDY+4+kO5HikTSeETPWE6HCDTJotqpAWISuwXpRr7bV7NqDbVlgK0qFSTobgOhkxc179G0j12fkQ60G/zViK8sDQliSiIqzgSAJS7CbEopqmCYzFN91EaSbdQY9L8qT15

In [17]:
# Inspecting the standardized AIMessage contract
print("Response Type  :", type(response_gemini))  # <class 'langchain_core.messages.ai.AIMessage'>
print("Content Type   :", type(response_gemini.content))
print("Finish Reason  :", response_gemini.response_metadata.get("finish_reason"))
print("Usage Metadata :", response_gemini.usage_metadata)

Response Type  : <class 'langchain_core.messages.ai.AIMessage'>
Content Type   : <class 'list'>
Finish Reason  : STOP
Usage Metadata : {'input_tokens': 8, 'output_tokens': 492, 'total_tokens': 500, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 456}}


### Deep Dive: Is `AIMessage` Unified Across All Providers?

**Yes, the interface contract is unified, but underlying provider nuances exist:**

| Attribute | Standard Contract | Provider Differences |
| :--- | :--- | :--- |
| **Object Type** | Always `AIMessage` (`type == "ai"`) | Identical across OpenAI, Anthropic, Gemini, Ollama. |
| **`.content`** | Model-generated text | Typically `str` (Claude, OpenAI), but can be `list[dict]` (Gemini multimodal SDK). |
| **`usage_metadata`** | `input_tokens`, `output_tokens`, `total_tokens` | Extra fields vary  |
| **`response_metadata`** | Raw response payload from provider | Stop reason keys differ: `finish_reason` (Gemini/OpenAI) vs `stop_reason` (Claude). |

> **Architectural Takeaway:**  
> Because `.content` can vary slightly between models (string vs. block list), production pipelines use **Output Parsers** (such as `StrOutputParser`) to enforce a guaranteed, pure string output across any model provider.

In [12]:
from langchain_core.messages import SystemMessage, HumanMessage

# Demonstrating Abstraction: The exact same message objects work across providers seamlessly
messages = [
    SystemMessage(content="You are an expert AI software architect. Reply concisely in one sentence."),
    HumanMessage(content="Why do we use abstraction layers when integrating LLMs in production?")
]

response = llm_gemini.invoke(messages)
print(response.content)

[{'type': 'text', 'text': ' vendor lock-in', 'extras': {'signature': 'EtMDCtADAWkUfRPSFeT5wJhWjVc6yJQh/6cP7hWdi79dH5lKV43Bj/oXbEd8yo4kvq8ikBBKULQCMeJh/+9K9g6+Pgh72sWu/zCdzWV3okLnIgwg8KkeYQYrLaFy8iX5ozQ1JuP1zCAWf1nOXjI2zUouIIePRgf17cNT4iKKBUWdrRAham1m2jx7IsYS268TqIAiSJX32u9x5yaG/bcCZO2Xc6UvYlCA8zXlSrkmVJAwVY5V1ScQN6qLxOxfR4ZRSOnGwUSjVnQNTG4PjYCta/ZmQt7XUEM7z1pYgIj6mlQdM9CAuOIwssoJvsd0xd+kTxWXTL+JXNASxtGiG1h2B2G0bQQ1//hJpaoD9ohjZkLmbQlJt4malX4/IEv0C1pVxcwIMN6gprw2FFGNaMrkfa39BVgLk3ISMuKM/F6ExaXlp3IAUVwFH4YJ1TgtvtUx90ULdisNn/f2nmhFAbYc/00OJ8f905eE60TkdVd8xpa8C08ZvdkVjeMhd2R/tisbpbIGuqZ62OYVt5rGFgQkbJCV+cqIbFghf2CzmrgEh2hqZ0oYkEhrojH4/R9fZK3gWkvXNLHRCnsM4zknOzQjD588EgwzjTj5msnwkaVdnGCryvMhNLE='}}]


## 4. Prompt Templates (`ChatPromptTemplate`)

Hardcoding static strings inside messages limits scalability in production applications. **`ChatPromptTemplate`** creates reusable, parameterized prompt templates that dynamically inject runtime values into structured message roles (`system`, `human`, `ai`).

```text
Input Variables (dict)
          │
          ▼
ChatPromptTemplate.invoke({...})
          │
          ▼
   ChatPromptValue
(Formatted Messages)
          │
          ▼
  ChatModel.invoke(...)
          │
          ▼
      AIMessage
```

### Shorthand Tuples Syntax in `from_messages`:
| Shorthand Tuple | Underlying Component | Purpose |
| :--- | :--- | :--- |
| `("system", "...")` | `SystemMessagePromptTemplate` | Defines assistant persona, tone, and system constraints. |
| `("human", "...")`  | `HumanMessagePromptTemplate`  | User queries with dynamic variables (`{topic}`). |
| `("ai", "...")`     | `AIMessagePromptTemplate`     | Assistant example turns for few-shot prompt construction. |

In [29]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Define parameterized ChatPromptTemplate using role tuples
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}."),
    ("human", "Explain {topic} to a {level} student in 2 clear sentences.")
])

# 2. Format template by passing runtime variables via .invoke()
formatted_prompt = prompt_template.invoke({
    "role": "Python teacher",
    "topic": "FastAPI",
    "level": "beginner"
})

# Inspect the formatted output (ChatPromptValue containing Message objects)
print("=== ChatPromptValue ===")
print(formatted_prompt)
print("=== listed Messages ===")
print(formatted_prompt.to_messages())
print("=== Formatted Messages ===")
for msg in formatted_prompt.to_messages():
    print(f"[{msg.type.upper()}]: {msg.content}")

=== ChatPromptValue ===
messages=[SystemMessage(content='You are a Python teacher.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain FastAPI to a beginner student in 2 clear sentences.', additional_kwargs={}, response_metadata={})]
=== listed Messages ===
[SystemMessage(content='You are a Python teacher.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain FastAPI to a beginner student in 2 clear sentences.', additional_kwargs={}, response_metadata={})]
=== Formatted Messages ===
[SYSTEM]: You are a Python teacher.
[HUMAN]: Explain FastAPI to a beginner student in 2 clear sentences.


In [37]:
# 3. Pass formatted prompt directly into the ChatModel
response = llm_gemini.invoke(formatted_prompt)

print("=== Model Response ===")
print(response.content)

=== Model Response ===
[{'type': 'text', 'text': '**FastAPI is a modern Python framework used to build APIs, which act as digital', 'extras': {'signature': 'EtMQCtAQAWkUfROjoFdfymfnUDSaCCDzFTg77xTRmjNgjBEI3OqB3rGvCGJTmCqiI+9bFgFnIGiMw3pzdtD51ZkKnWbcQGjtO5kGNQrqtOxIGneQsFHgANqu6DKoSMnwBLvDMD4yzLBYAkPd610aImjYZRM1kUFeFcWEPA+51nb3vsebPVDa3b2tkCJkryonDponkGfJYek4/mpXlB+/NdUOeuGtuHSlVR6ClKpaZYKCYeOCnnyPLSVAOeoWh8CrBsDCrzZ/ySZiRZKNw+OfblZR3rrQM1BY3ygPgMfFtWsnVjueHFrtWniurUin5/xU9xNYIheJeO1EhjQedB3GQRBkDLQX0ixs13aOw4dEf5yNJspYisDdapvR8rPD7mrI9Svka2aIJyrSUSriZlJ1qThXhL6/Fwenx+HClH/De4EsusRubh/+IfozrASM81oWMIG3CFI60+rNck7t+jsBBiG5OzTvno6ldZGI23a7pfDMnm87RBIzlfz86kN80L9zmkfufBVBH30cy8SActkvHvqkiZ0JgHM62VxdT4sOQB1nLFm9YjzvxurkfdJ3bCYiPsJuw4V8wcnk16lQ3YVDZG9YyYslGUgY6npSZN5CGCX3BuM5crT+wUQchG52HaE72ZA5mwCxI++gpVz9tJkGALUFGYUYCLfv/NAAHrplSMHdKHqopY5cKvegfCkOAi1+Gu2DmJ/5aoIf5JVCzk8zvKLBiQ3B/ipOZewW3nwORLHR/D8IOUkBW68gkWA3CK7iFGj80KnG+EHrfvY/AIKtwTeOLdnFKF0x9TlUOCU0FGxWUTyFMJlce2slfhM43EXruOsqGoHlrLe

## 5. The Runnable Protocol & LCEL (LangChain Expression Language)

At the core of LangChain's architecture is the **Runnable Protocol**. A `Runnable` is the standardized base class / contract that every pipeline component inherits from.

### Architectural Clarity: Component vs. Interface vs. Runnable

| Concept | Definition | Concrete Example |
| :--- | :--- | :--- |
| **Component** | The functional modular unit | `ChatPromptTemplate`, `ChatAnthropic` |
| **Interface** | Standardized execution methods | `.invoke()`, `.ainvoke()`, `.stream()` |
| **Runnable** | The base class / contract inherited by components | `isinstance(prompt, Runnable) == True` |
| **Chain** | Pipeline composed via the pipe operator (`|`) | `chain = prompt | llm` (`RunnableSequence`) |

---

### LCEL Pipe Operator (`|`)
LangChain Expression Language (LCEL) uses the Unix pipe operator (`|`) to compose individual Runnables into a unified sequential pipeline (a **`RunnableSequence`**).

```text
Input Dict ──▶ [ ChatPromptTemplate ] ──▶ ChatPromptValue ──▶ [ ChatModel ] ──▶ AIMessage
                       │                                             │
                       └────────────────── Pipe (|) ─────────────────┘
```
The output of each component automatically becomes the input of the next component.

In [ ]:
from langchain_core.runnables import RunnableLambda

# 1. Define standard Python functions
def double(x: int) -> int:
    return x * 2

def add_one(x: int) -> int:
    return x + 1

# 2. RunnableLambda wraps ordinary functions into standard Runnable components
double_runnable = RunnableLambda(double)
add_one_runnable = RunnableLambda(add_one)

# --- Method 1: Manual Sequential Execution (Without Pipe |) ---
step1_result = double_runnable.invoke(5)         # 5 * 2 = 10
step2_result = add_one_runnable.invoke(step1_result) # 10 + 1 = 11

print("=== 1. Manual Execution (Without |) ===")
print("Step 1 (double)  :", step1_result)
print("Step 2 (add_one) :", step2_result)

# --- Method 2: LCEL Composition (With Pipe |) ---
math_chain = double_runnable | add_one_runnable
chain_result = math_chain.invoke(5) # 11

print("\n=== 2. LCEL Chain Execution (With |) ===")
print("Chain Type       :", type(math_chain))  # <class 'langchain_core.runnables.base.RunnableSequence'>
print("Chain Result     :", chain_result)


=== 1. Manual Execution (Without |) ===
Step 1 (double)  : 10
Step 2 (add_one) : 11

=== 2. LCEL Chain Execution (With |) ===
Chain Type       : <class 'langchain_core.runnables.base.RunnableSequence'>
Chain Result     : 11


In [45]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Define reusable prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {role}."),
    ("human", "Explain {topic} to a {level} student in 2 clear sentences.")
])

# 2. Compose end-to-end LCEL Chain (Prompt -> ChatModel)
# Input dict automatically flows through: dict -> PromptTemplate -> ChatModel -> AIMessage
chain = prompt | llm

# 3. Single .invoke() executes the full chain seamlessly
response = chain.invoke({
    "role": "Python teacher",
    "topic": "FastAPI",
    "level": "beginner"
})

print("Chain Structure :", chain)
print("\n=== Model Response ===")
print(response.content)

Chain Structure : first=ChatPromptTemplate(input_variables=['level', 'role', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['role'], input_types={}, partial_variables={}, template='You are an expert {role}.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level', 'topic'], input_types={}, partial_variables={}, template='Explain {topic} to a {level} student in 2 clear sentences.'), additional_kwargs={})]) middle=[] last=ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain-anthropic': '1.7.5'}}, profile={'name': 'Claude Haiku 4.5', 'release_date': '2025-10-15', 'last_updated': '2025-10-15', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 64000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False,

## 6. Composition Primitives: `RunnablePassthrough` & `RunnableParallel`

In real-world applications (especially **RAG** pipelines), processing is rarely strictly linear. You often need to **branch** execution across multiple parallel tasks, or **forward unaltered inputs** to downstream components.

```text
                               ┌──▶ Runnable A (e.g. Fetch Context) ──▶ Output A
                               │
Input ──▶ [ RunnableParallel ] ┤
                               │
                               └──▶ [ RunnablePassthrough ] ──────────▶ Original Input
```

| Primitive | Purpose | Typical Role in Pipelines |
| :--- | :--- | :--- |
| **`RunnablePassthrough`** | Passes its input through unmodified (`f(x) = x`) | Preserves the raw user query when fetching external context. |
| **`RunnableParallel`** | Executes multiple runnables concurrently on the same input | Branches execution and packages results into a structured dictionary. |

In [48]:
from langchain_core.runnables import RunnablePassthrough

# 1. RunnablePassthrough acts as an identity function (forwards input unchanged)
passthrough = RunnablePassthrough()

result = passthrough.invoke("Hello, LangChain!")
print("Input forwarded as-is:", result)

Input forwarded as-is: Hello, LangChain!


In [49]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

# 1. Define standard functions
double_runnable = RunnableLambda(lambda x: x * 2)
add_one_runnable = RunnableLambda(lambda x: x + 1)

# 2. RunnableParallel executes both branches concurrently on the same input (5)
parallel_chain = RunnableParallel(
    double=double_runnable,
    add_one=add_one_runnable
)

parallel_output = parallel_chain.invoke(5)
print("Parallel Output (dict):", parallel_output)  # {'double': 10, 'add_one': 6}

Parallel Output (dict): {'double': 10, 'add_one': 6}


In [50]:
# 3. The Classic RAG Pattern: Combining Passthrough & Parallel
# Forwards the original input alongside derived calculations in a single dictionary
branching_pipeline = RunnableParallel(
    original_input=RunnablePassthrough(),
    doubled=double_runnable,
    incremented=add_one_runnable
)

final_dict = branching_pipeline.invoke(10)
print("Branching Pipeline Output:")
print(final_dict)  # {'original_input': 10, 'doubled': 20, 'incremented': 11}

Branching Pipeline Output:
{'original_input': 10, 'doubled': 20, 'incremented': 11}


## 7. Data Representation: `Document` Objects

In RAG (Retrieval-Augmented Generation) applications, text cannot be processed in isolation—it must carry contextual metadata (such as source file, page number, creation date, or section) to enable precise filtering, citation, and provenance tracking.

LangChain encapsulates this inside the standard **`Document`** container:

```text
Document Object
   │
   ├── .page_content   ──▶ Raw text content (string)
   └── .metadata       ──▶ Key-value dictionary containing document provenance (dict)
```

| Attribute | Type | Purpose |
| :--- | :--- | :--- |
| **`page_content`** | `str` | The actual text body to be chunked, embedded, and retrieved. |
| **`metadata`** | `dict` | Contextual metadata dictionary (e.g., `source`, `page`, `author`). |

In [7]:
from langchain_core.documents import Document

# Creating a Document object manually with content and metadata
doc = Document(
    page_content="FastAPI is a modern Python web framework designed for high-performance APIs.",
    metadata={"source": "fastapi_guide.pdf", "page": 3, "topic": "Backend"}
)

print("Page Content :", doc.page_content)
print("Metadata     :", doc.metadata)
print("Source File  :", doc.metadata.get("source"))

Page Content : FastAPI is a modern Python web framework designed for high-performance APIs.
Metadata     : {'source': 'fastapi_guide.pdf', 'page': 3, 'topic': 'Backend'}
Source File  : fastapi_guide.pdf


## 8. Package Ecosystem & Document Loaders

### The LangChain Monorepo Architecture
To prevent bloated installations and decouple dependencies, LangChain splits its codebase into specialized packages:

```text
                         LangChain Framework
                                  │
    ┌─────────────────────────────┼─────────────────────────────┐
    ↓                             ↓                             ↓
langchain-core               langchain                langchain-community
(Base Primitives)     (Orchestration & Chains)       (Third-Party Integrations)
  ├── Runnables                 ├── Prebuilt Chains            ├── Document Loaders (PDF, TXT, Web)
  ├── Messages                  ├── Agents                     ├── Vector Stores (Qdrant, Chroma)
  ├── Prompts                   └── Memory                     └── Embeddings & Tools
  └── Documents
                                  │
                                  ▼
                          Provider Packages
                      ├── langchain-anthropic (Claude)
                      ├── langchain-google-genai (Gemini)
                      └── langchain-openai (GPT models)
```

| Package | Role & Contents | Typical Imports |
| :--- | :--- | :--- |
| **`langchain-core`** | Zero-dependency base contracts and protocols | `Runnable`, `HumanMessage`, `ChatPromptTemplate`, `Document` |
| **`langchain-community`** | Community-maintained third-party integrations | `TextLoader`, `PyPDFLoader`, `Qdrant`, `WebBaseLoader` |
| **`langchain`** | High-level orchestrations, prebuilt chains, and agents | `create_retrieval_chain`, agent executors |
| **`langchain-<provider>`** | Dedicated, optimized vendor SDK adapters | `ChatAnthropic`, `ChatGoogleGenerativeAI` |

```bash
# Install community connectors (document loaders, vector stores)
uv add langchain-community
```

---

### What is a Document Loader?
A **Document Loader** is a standardized component that ingests data from external sources (TXT, PDF, Word, HTML, Webpages, Notion, Databases) and parses them into a list of standardized LangChain `Document` objects (`list[Document]`).

```text
External Source (file / web / db)
               │
               ▼
      [ DocumentLoader ]
               │ (.load() / .lazy_load())
               ▼
        list[Document]
  [ Document(page_content, metadata) ]
```

In [8]:
from langchain_community.document_loaders import TextLoader

# 1. Initialize the loader with the target document path
loader = TextLoader("article.txt")

# 2. Ingest the file and parse it into Document objects
documents = loader.load()

# 3. Inspect the loaded Document container structure
print("Container Type :", type(documents))     # <class 'list'>
print("Document Count :", len(documents))      # 1 Document for this text file
print("Element Type   :", type(documents[0]))  # <class 'langchain_core.documents.base.Document'>

print("\n=== Document Metadata ===")
print(documents[0].metadata)  # {'source': 'article.txt'}

print("\n=== Content Preview (First 250 characters) ===")
print(documents[0].page_content[:250] + "...")

Container Type : <class 'list'>
Document Count : 1
Element Type   : <class 'langchain_core.documents.base.Document'>

=== Document Metadata ===
{'source': 'article.txt'}

=== Content Preview (First 250 characters) ===
Welcome to Wikipedia! Creating a new article is one of many ways to contribute to the encyclopedia, but can be difficult for new editors. We recommend looking at our introductory tutorials or contributing to Wikipedia to learn the basics about editin...


### 8.2 PDF Loader (`PyMuPDFLoader`)

Loads PDF files into LangChain `Document` objects (one `Document` per page).

```bash
uv add pymupdf
```

- **`.load()` / `.aload()`**: Loads all pages into a list at once.
- **`.lazy_load()`**: Streams pages one-by-one as an iterator (saves memory).

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

# 1. Load entire PDF into a list of Documents
loader = PyMuPDFLoader("Document_31.pdf")
documents = await loader.aload()

print("Total Pages :", len(documents))
print("Metadata    :", documents[0].metadata)
print("\nFirst Page Content:\n", documents[0].page_content[:200])

/var/folders/q1/74snd04j3rl985w_vly9qg3h0000gn/T/ipykernel_14025/3006845958.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Total Pages : 5
Metadata    : {'producer': '', 'creator': 'Microsoft Word', 'creationdate': '2026-04-27T18:18:21-07:00', 'source': 'Document_31.pdf', 'file_path': 'Document_31.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'fatima osama slsaed mohammd', 'subject': '', 'keywords': '', 'moddate': '2026-04-27T18:18:21-07:00', 'trapped': '', 'modDate': "D:20260427181821-07'00'", 'creationDate': "D:20260427181821-07'00'", 'page': 0}

First Page Content:
 #   
 تقرير األمن السيبراني الشامل- نظام EchoMind ()ملخص مختصر 
 
 نظرة عامة## 
 
تم تنفيذ نظام أمني متعدد الطبقات (Defense in Depth)  لحماية بيانات المستخدمين والصور الطبية الحساسة، يعتمد على
.مجموعة


In [14]:
# 2. Lazy loading page-by-page (Iterator)
for doc in loader.lazy_load():
    print("Page:", doc.metadata.get("page"), "| Characters:", len(doc.page_content))

Page: 0 | Characters: 1761
Page: 1 | Characters: 1594
Page: 2 | Characters: 1756
Page: 3 | Characters: 1282
Page: 4 | Characters: 1215


## 9. Text Chunking & Splitting (`langchain_text_splitters`)

Raw documents are typically too large to embed effectively or fit into LLM context windows. **Chunking** divides long texts into smaller, semantically coherent passages.

---

### The `RecursiveCharacterTextSplitter` Algorithm
`RecursiveCharacterTextSplitter` is the standard, recommended splitter in LangChain. It splits text using a hierarchical list of separators—preserving larger semantic units (paragraphs, lines) whenever possible, and only falling back to smaller units (words, characters) when a chunk exceeds `chunk_size`.

```text
Hierarchical Separator Order:
1. "\n\n"   ──▶ Paragraph boundaries (Preserves semantic topic flow)
2. "\n"     ──▶ Line breaks
3. " "      ──▶ Word boundaries (Avoids cutting words in half)
4. ""       ──▶ Character level (Last resort fallback)
```

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`chunk_size`** | `int` | `4000` | Maximum character length of each chunk. |
| **`chunk_overlap`** | `int` | `200` | Shared characters between adjacent chunks to maintain context across boundaries (**must be < `chunk_size`**). |

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Demonstrating chunking & overlap mechanism on raw string
text = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=3  # 3 overlapping characters between consecutive chunks
)

chunks = splitter.split_text(text)

print("=== Overlap Demonstration (split_text) ===")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {repr(chunk)}") #Representation return the object

=== Overlap Demonstration (split_text) ===
Chunk 0: 'ABCDEFGHIJ'
Chunk 1: 'HIJKLMNOPQ'
Chunk 2: 'OPQRSTUVWX'
Chunk 3: 'VWXYZ'


### `split_text()` vs. `split_documents()`

| Method | Input | Output | Metadata Handling |
| :--- | :--- | :--- | :--- |
| **`split_text()`** | Raw string (`str`) | `list[str]` | None (returns plain strings) |
| **`split_documents()`** | `list[Document]` | `list[Document]` | **Preserves and propagates metadata** from the parent document to each child chunk |

In [28]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)  # return chunk =list[Document] type not str has page_content and metadata

print("Original documents:", len(documents))
print("Chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(
        f"Chunk {i} | "
        f"Characters: {len(chunk.page_content)} | "
        f"Metadata: {chunk.metadata}"
    )

print(type(chunks[0]))
print(repr(chunks[0]))


Original documents: 5
Chunks: 19
Chunk 0 | Characters: 399 | Metadata: {'producer': '', 'creator': 'Microsoft Word', 'creationdate': '2026-04-27T18:18:21-07:00', 'source': 'Document_31.pdf', 'file_path': 'Document_31.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'fatima osama slsaed mohammd', 'subject': '', 'keywords': '', 'moddate': '2026-04-27T18:18:21-07:00', 'trapped': '', 'modDate': "D:20260427181821-07'00'", 'creationDate': "D:20260427181821-07'00'", 'page': 0}
Chunk 1 | Characters: 409 | Metadata: {'producer': '', 'creator': 'Microsoft Word', 'creationdate': '2026-04-27T18:18:21-07:00', 'source': 'Document_31.pdf', 'file_path': 'Document_31.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'fatima osama slsaed mohammd', 'subject': '', 'keywords': '', 'moddate': '2026-04-27T18:18:21-07:00', 'trapped': '', 'modDate': "D:20260427181821-07'00'", 'creationDate': "D:20260427181821-07'00'", 'page': 0}
Chunk 2 | Characters: 474 | Metadata: {'produ

### TokenTextSplitter
- Default tokenizer: `tiktoken`

```text
TokenTextSplitter
        │
        └── Accurate in token count, but less attentive to text structure


RecursiveCharacterTextSplitter
        │
        └── Preserves text structure better, but chunk_size is based on characters
```

In [8]:
from langchain_text_splitters import TokenTextSplitter

splitter = TokenTextSplitter(
    chunk_size=100,  # 100 tokens, not characters
    chunk_overlap=10
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks[0:16]):
    print(
        f"Chunk {i} | "
        f"Characters: {len(chunk.page_content)}"
    )


Chunk 0 | Characters: 105
Chunk 1 | Characters: 117
Chunk 2 | Characters: 118
Chunk 3 | Characters: 111
Chunk 4 | Characters: 111
Chunk 5 | Characters: 99
Chunk 6 | Characters: 106
Chunk 7 | Characters: 107
Chunk 8 | Characters: 98
Chunk 9 | Characters: 116
Chunk 10 | Characters: 104
Chunk 11 | Characters: 101
Chunk 12 | Characters: 101
Chunk 13 | Characters: 107
Chunk 14 | Characters: 102
Chunk 15 | Characters: 106


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def fake_token_length(text):
    # return len(tokenizer.encode(text, add_special_tokens=False))
    return len(text.split())

splitter = RecursiveCharacterTextSplitter(
    chunk_size=20,
    chunk_overlap=5,
    length_function=fake_token_length # word instead token
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks[:5]):
    print(
        f"Chunk {i}"
        f"\nWords/Tokens: {fake_token_length(chunk.page_content)}"
        f"\nCharacters: {len(chunk.page_content)}"
        f"\nContent: {chunk.page_content[:100]}"
        f"\n{'-' * 50}"
    )


Chunk 0
Words/Tokens: 11
Characters: 78
Content: #   
 تقرير األمن السيبراني الشامل- نظام EchoMind ()ملخص مختصر 
 
 نظرة عامة##
--------------------------------------------------
Chunk 1
Words/Tokens: 19
Characters: 125
Content: نظرة عامة## 
 
تم تنفيذ نظام أمني متعدد الطبقات (Defense in Depth)  لحماية بيانات المستخدمين والصور 
--------------------------------------------------
Chunk 2
Words/Tokens: 17
Characters: 126
Content: .مجموعة من الخوارزميات والتقنيات األمنية المتكاملة 
 
--- 
 
 ##
1
 . خوارزميات التشفير المستخدمة 
 
--------------------------------------------------
Chunk 3
Words/Tokens: 16
Characters: 105
Content: **AES-128  عبر Fernet** - 
 خوارزمية تشفير متماثل معتمدة من الحكومة األمريكية، تستخدم مفتاح128
 بت ف
--------------------------------------------------
Chunk 4
Words/Tokens: 18
Characters: 119
Content: بت في وضع 
CBC. تم تنفيذها باستخدام مكتبة cryptography.fernet،  ويتم تخزين المفتاح في متغيرات البيئة
--------------------------------------------------


### 9.4 Architectural Trade-offs in Choosing `chunk_size`

```text
                  chunk_size
                      │
            ┌─────────┴─────────┐
            ↓                   ↓
     Model Limitation    Retrieval Quality
     (Hard Constraint)   (Soft Constraint)
     `max_seq_length`    Information Density
```

| Factor | Description | Failure Mode if Violated |
| :--- | :--- | :--- |
| **Model Limitation** (`max_seq_length`) | Strict token ceiling of the embedding model (e.g., 256 for `all-MiniLM-L6-v2`, 512 for `e5-base`, 8192 for `bge-m3`). | **Silent Truncation:** Tokens beyond the limit are permanently dropped during embedding, losing critical data. |
| **Retrieval Quality** (Information Density) | The semantic completeness of each chunk required to answer queries accurately. | **Context Loss vs. Dilution:** Too small loses context; too large dilutes vector similarity with noise. |

---

#### The Sizing Spectrum: Too Small vs. Too Large

```text
  Too Small (< 50 tokens)                  Optimal (150 - 500 tokens)                 Too Large (> 1000 tokens)
───────────┼───────────────────────────────────────────┼─────────────────────────────────────────┼───────────
 • Misses surrounding context               • Coherent semantic units                 • Vector Dilution (multi-topic noise)
 • Fragmented sentence fragments            • High similarity search precision        • Inefficient context window usage
 • High database point overhead             • Standalone, answerable meaning          • May exceed embedding max_seq_length
```